# 03 — Baselines S2 e atributos S3

Dummy com prior e regressão logística simples (#8). O encoder e a escala são ajustados somente no treino. Não há tuning ou escolha do modelo final nesta etapa.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display
from src.data import clean, load_raw
from src.features import build_features, add_patient_history
from src.models import split_by_patient
from src.analysis import rate_table
import pandas as pd
import matplotlib.pyplot as plt
all_data = add_patient_history(build_features(clean(load_raw(ROOT / 'data/raw/KaggleV2-May-2016.csv'))))
train_idx, test_idx = split_by_patient(all_data)
eda = all_data.iloc[train_idx].copy()


In [ ]:
from src.models import run_baselines, split_temporal
results = run_baselines(all_data, train_idx, test_idx)
display(pd.DataFrame(results).T)
assert not set(all_data.iloc[train_idx].patient_id) & set(all_data.iloc[test_idx].patient_id)

## Checagem temporal

Corte em 25/05/2016. O treino só usa consultas anteriores ao corte; o teste só usa agendamentos a partir dele. Pacientes podem se repetir nesta checagem. Histórico pode incorporar desfechos anteriores ao novo agendamento, como aconteceria em uso sequencial.

In [ ]:
temporal_train, temporal_test = split_temporal(all_data)
display(pd.DataFrame(run_baselines(all_data, temporal_train, temporal_test)).T)
print('Treino e teste temporal:', len(temporal_train), len(temporal_test))

## Interpretação

PR-AUC é reportada como average precision (AP). Matriz de confusão tem linhas reais e colunas preditas, na ordem compareceu/faltou. O limiar padrão é 0,5; o top 20% avalia ranking independentemente desse limiar. Empates recebem captura esperada sob desempate aleatório. O dummy deve alcançar AP próxima da prevalência e recall@20% de 20%. Veja os números e as limitações em [sprints-2-3.md](../docs/sprints-2-3.md).